# Tema 5 — Hoja de Ejercicios IV
# Embeddings contextualizados y similitud semántica

Este notebook resuelve en un único archivo la **Hoja de Ejercicios IV**.

Trabajaremos con:

- **BERT** para obtener embeddings contextualizados a nivel de token.
- **Mean pooling** y **max pooling** para construir vectores de frase a partir de BERT.
- **SBERT** para obtener embeddings directamente optimizados para comparación de frases.
- **Word2Vec** como ejemplo de embeddings estáticos.
- Comparación de similitud coseno en frases con significado parecido, polisemia y cambios sutiles de significado.

La idea principal es comparar cómo cambia la similitud entre frases según el tipo de embedding utilizado.

## 0. Instalación de librerías

En Google Colab puedes ejecutar esta celda si no tienes instaladas las librerías.

Si trabajas en local con `uv`, puedes usar:

```bash
uv add transformers sentence-transformers scikit-learn torch numpy pandas gensim
```

**Nota sobre Word2Vec:** el modelo `word2vec-google-news-300` es muy grande. Puede tardar bastante en descargarse y consumir mucha memoria. En el notebook se deja preparado, pero comentado.

In [5]:
# En Colab, descomenta esta línea si hace falta:
# !pip install -q transformers sentence-transformers scikit-learn torch numpy pandas gensim

## 1. Importación de librerías

In [6]:
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

Dispositivo: cpu


## 2. Funciones auxiliares

La similitud coseno mide el parecido entre dos vectores.

- Valor cercano a `1`: vectores muy parecidos.
- Valor cercano a `0`: vectores poco relacionados.
- Valor negativo: direcciones opuestas.

In [7]:
def cosine_sim(vec1, vec2):
    vec1 = np.array(vec1).reshape(1, -1)
    vec2 = np.array(vec2).reshape(1, -1)
    return cosine_similarity(vec1, vec2)[0][0]


def build_similarity_table(query, sentences, query_embedding, sentence_embeddings):
    rows = []
    for sentence, emb in zip(sentences, sentence_embeddings):
        rows.append({
            "Consulta": query,
            "Frase comparada": sentence,
            "Similitud coseno": cosine_sim(query_embedding, emb)
        })
    return pd.DataFrame(rows).sort_values("Similitud coseno", ascending=False)


def pairwise_similarity_table(sentences, embeddings):
    sim_matrix = cosine_similarity(embeddings)
    return pd.DataFrame(sim_matrix, index=sentences, columns=sentences)

# Ejercicio 1. Embeddings contextualizados con BERT

El ejercicio pide usar `bert-base-uncased` para obtener embeddings contextualizados.

Frase de consulta:

```text
Dogs are domestic animals.
```

Frases a comparar:

```text
Dogs are pets.
This is a dog.
They are free today.
```

Calcularemos dos representaciones de frase:

1. **Mean pooling**: media de los embeddings de los tokens.
2. **Max pooling**: máximo por dimensión de los embeddings de los tokens.

Además, eliminaremos tokens especiales como `[CLS]`, `[SEP]` y `[PAD]` antes de hacer el pooling.

In [8]:
query = "Dogs are domestic animals."

sentences = [
    "Dogs are pets.",
    "This is a dog.",
    "They are free today."
]

all_texts = [query] + sentences
all_texts

['Dogs are domestic animals.',
 'Dogs are pets.',
 'This is a dog.',
 'They are free today.']

## 1.1 Cargar BERT y su tokenizador

`bert-base-uncased` es un modelo encoder. La salida `last_hidden_state` contiene un vector contextualizado para cada token.

In [9]:
bert_model_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(device)
bert_model.eval()

print("Modelo BERT cargado:", bert_model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo BERT cargado: bert-base-uncased


## 1.2 Obtener embeddings de BERT

La forma de `last_hidden_state` es:

```text
(batch_size, sequence_length, hidden_size)
```

Para `bert-base-uncased`, cada token se representa con un vector de 768 dimensiones.

In [10]:
def get_bert_token_embeddings(text, tokenizer, model, max_length=64):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=max_length
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    input_ids = encoded["input_ids"][0].cpu().numpy()
    attention_mask = encoded["attention_mask"][0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    hidden_states = outputs.last_hidden_state[0].cpu().numpy()

    return tokens, input_ids, attention_mask, hidden_states

## 1.3 Eliminar tokens especiales

El enunciado recomienda eliminar `[CLS]`, `[SEP]` y `[PAD]` al calcular el vector promedio.

Esto evita que el embedding final de la frase esté contaminado por tokens que no son palabras reales de la oración.

In [11]:
SPECIAL_TOKENS = {"[CLS]", "[SEP]", "[PAD]"}


def filter_real_token_embeddings(tokens, attention_mask, hidden_states):
    real_tokens = []
    real_embeddings = []

    for token, mask_value, embedding in zip(tokens, attention_mask, hidden_states):
        if mask_value == 1 and token not in SPECIAL_TOKENS:
            real_tokens.append(token)
            real_embeddings.append(embedding)

    return real_tokens, np.array(real_embeddings)


def bert_sentence_embedding(text, pooling="mean"):
    tokens, input_ids, attention_mask, hidden_states = get_bert_token_embeddings(
        text, bert_tokenizer, bert_model
    )

    real_tokens, real_embeddings = filter_real_token_embeddings(
        tokens, attention_mask, hidden_states
    )

    if pooling == "mean":
        sentence_embedding = np.mean(real_embeddings, axis=0)
    elif pooling == "max":
        sentence_embedding = np.max(real_embeddings, axis=0)
    else:
        raise ValueError("pooling debe ser 'mean' o 'max'")

    return sentence_embedding, real_tokens

## 1.4 Inspeccionar tokens reales

In [12]:
for text in all_texts:
    emb, real_tokens = bert_sentence_embedding(text, pooling="mean")
    print("Texto:", text)
    print("Tokens reales:", real_tokens)
    print("Dimensión del embedding:", emb.shape)
    print("-" * 80)

Texto: Dogs are domestic animals.
Tokens reales: ['dogs', 'are', 'domestic', 'animals', '.']
Dimensión del embedding: (768,)
--------------------------------------------------------------------------------
Texto: Dogs are pets.
Tokens reales: ['dogs', 'are', 'pets', '.']
Dimensión del embedding: (768,)
--------------------------------------------------------------------------------
Texto: This is a dog.
Tokens reales: ['this', 'is', 'a', 'dog', '.']
Dimensión del embedding: (768,)
--------------------------------------------------------------------------------
Texto: They are free today.
Tokens reales: ['they', 'are', 'free', 'today', '.']
Dimensión del embedding: (768,)
--------------------------------------------------------------------------------


## 1.5 Mean pooling con BERT

In [13]:
query_bert_mean, _ = bert_sentence_embedding(query, pooling="mean")
sentences_bert_mean = [bert_sentence_embedding(s, pooling="mean")[0] for s in sentences]

bert_mean_results = build_similarity_table(query, sentences, query_bert_mean, sentences_bert_mean)
bert_mean_results

,Consulta,Frase comparada,Similitud coseno
0,Dogs are domestic animals.,Dogs are pets.,0.817135
1,Dogs are domestic animals.,This is a dog.,0.697745
2,Dogs are domestic animals.,They are free today.,0.570568


## 1.6 Max pooling con BERT

In [14]:
query_bert_max, _ = bert_sentence_embedding(query, pooling="max")
sentences_bert_max = [bert_sentence_embedding(s, pooling="max")[0] for s in sentences]

bert_max_results = build_similarity_table(query, sentences, query_bert_max, sentences_bert_max)
bert_max_results

,Consulta,Frase comparada,Similitud coseno
1,Dogs are domestic animals.,This is a dog.,0.811080
0,Dogs are domestic animals.,Dogs are pets.,0.798324
2,Dogs are domestic animals.,They are free today.,0.758019


## 1.7 Comparación mean pooling vs max pooling

In [15]:
comparison_bert = pd.DataFrame({
    "Frase": sentences,
    "BERT mean pooling": [cosine_sim(query_bert_mean, emb) for emb in sentences_bert_mean],
    "BERT max pooling": [cosine_sim(query_bert_max, emb) for emb in sentences_bert_max]
}).sort_values("BERT mean pooling", ascending=False)

comparison_bert

,Frase,BERT mean pooling,BERT max pooling
0,Dogs are pets.,0.817135,0.798324
1,This is a dog.,0.697745,0.811080
2,They are free today.,0.570568,0.758019


## Conclusión del Ejercicio 1

BERT genera embeddings contextualizados a nivel de token. Para obtener un embedding de frase hemos usado pooling.

- **Mean pooling** suele ser más estable.
- **Max pooling** puede resaltar dimensiones concretas, pero puede ser más ruidoso.

Lo esperable es que las frases sobre perros sean más similares a `Dogs are domestic animals.` que `They are free today.`. Aun así, BERT base no está optimizado específicamente para similitud entre frases completas.

# Ejercicio 2. Embeddings contextualizados con SBERT

SBERT está diseñado para obtener embeddings de frases directamente.

Usaremos:

- `sentence-transformers/all-MiniLM-L6-v2`
- `sentence-transformers/all-mpnet-base-v2`

## 2.1 Modelo `all-MiniLM-L6-v2`

In [16]:
sbert_minilm_name = "sentence-transformers/all-MiniLM-L6-v2"
sbert_minilm = SentenceTransformer(sbert_minilm_name, device=device)
print("Modelo cargado:", sbert_minilm_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\hugo\PycharmProjects\learn-advanced-nlp-deep-learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hugo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado: sentence-transformers/all-MiniLM-L6-v2


In [31]:
query_minilm = sbert_minilm.encode(query)
sentences_minilm = sbert_minilm.encode(sentences)

minilm_results = build_similarity_table(query, sentences, query_minilm, sentences_minilm)
minilm_results

,Consulta,Frase comparada,Similitud coseno
0,Dogs are domestic animals.,Dogs are pets.,0.848573
1,Dogs are domestic animals.,This is a dog.,0.554901
2,Dogs are domestic animals.,They are free today.,0.097927


## 2.2 Modelo `all-mpnet-base-v2`

In [33]:
sbert_mpnet_name = "sentence-transformers/all-mpnet-base-v2"
sbert_mpnet = SentenceTransformer(sbert_mpnet_name, device=device)
print("Modelo cargado:", sbert_mpnet_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado: sentence-transformers/all-mpnet-base-v2


In [34]:
query_mpnet = sbert_mpnet.encode(query)
sentences_mpnet = sbert_mpnet.encode(sentences)

mpnet_results = build_similarity_table(query, sentences, query_mpnet, sentences_mpnet)
mpnet_results

,Consulta,Frase comparada,Similitud coseno
0,Dogs are domestic animals.,Dogs are pets.,0.835124
1,Dogs are domestic animals.,This is a dog.,0.482438
2,Dogs are domestic animals.,They are free today.,0.068279


## 2.3 Comparación entre modelos SBERT

In [35]:
comparison_sbert = pd.DataFrame({
    "Frase": sentences,
    "SBERT all-MiniLM-L6-v2": [cosine_sim(query_minilm, emb) for emb in sentences_minilm],
    "SBERT all-mpnet-base-v2": [cosine_sim(query_mpnet, emb) for emb in sentences_mpnet]
}).sort_values("SBERT all-mpnet-base-v2", ascending=False)

comparison_sbert

,Frase,SBERT all-MiniLM-L6-v2,SBERT all-mpnet-base-v2
0,Dogs are pets.,0.848573,0.835124
1,This is a dog.,0.554901,0.482438
2,They are free today.,0.097927,0.068279


## 2.4 Otros modelos SBERT inspirados en MTEB

El benchmark MTEB compara muchos modelos de embeddings. Algunos modelos que puedes probar son:

- `sentence-transformers/all-MiniLM-L12-v2`
- `sentence-transformers/paraphrase-MiniLM-L6-v2`
- `sentence-transformers/multi-qa-MiniLM-L6-cos-v1`
- `BAAI/bge-small-en-v1.5`

La celda siguiente permite probar modelos adicionales.

In [36]:
extra_sbert_models = [
    "sentence-transformers/all-MiniLM-L12-v2",
    "sentence-transformers/paraphrase-MiniLM-L6-v2",
    # "BAAI/bge-small-en-v1.5",
]

extra_results = []

for model_name in extra_sbert_models:
    print("Cargando:", model_name)
    model_extra = SentenceTransformer(model_name, device=device)
    q_emb = model_extra.encode(query)
    s_embs = model_extra.encode(sentences)

    for sentence, emb in zip(sentences, s_embs):
        extra_results.append({
            "Modelo": model_name,
            "Frase": sentence,
            "Similitud coseno": cosine_sim(q_emb, emb)
        })

df_extra_sbert = pd.DataFrame(extra_results)
df_extra_sbert.sort_values(["Modelo", "Similitud coseno"], ascending=[True, False])

Cargando: sentence-transformers/all-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cargando: sentence-transformers/paraphrase-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Modelo,Frase,Similitud coseno
0,sentence-transformers/all-MiniLM-L12-v2,Dogs are pets.,0.866089
1,sentence-transformers/all-MiniLM-L12-v2,This is a dog.,0.568648
2,sentence-transformers/all-MiniLM-L12-v2,They are free today.,0.083869
3,sentence-transformers/paraphrase-MiniLM-L6-v2,Dogs are pets.,0.836309
4,sentence-transformers/paraphrase-MiniLM-L6-v2,This is a dog.,0.405211
5,sentence-transformers/paraphrase-MiniLM-L6-v2,They are free today.,-0.030277


## Conclusión del Ejercicio 2

SBERT suele ser más adecuado que BERT base para comparar frases completas porque está entrenado específicamente para generar embeddings de frase útiles en similitud semántica.

- `all-MiniLM-L6-v2`: rápido y ligero.
- `all-mpnet-base-v2`: más pesado, pero suele ofrecer mejor calidad.

# Ejercicio 3. Comparativa de embeddings

Se comparan:

1. Word2Vec con media de palabras.
2. BERT con embeddings contextualizados y pooling.
3. SBERT con embeddings de frase.

La pregunta principal es qué opción mide mejor la similitud entre frases.

## 3.1 Word2Vec

Word2Vec genera embeddings estáticos. Una palabra tiene el mismo vector aunque aparezca en contextos diferentes.

El modelo `word2vec-google-news-300` es muy grande. Por eso la descarga queda comentada.

In [41]:
import gensim.downloader as api
word2vec_model = api.load("word2vec-google-news-300")
print("Word2Vec cargado.")

Word2Vec cargado.


In [43]:
def simple_tokenize_for_word2vec(text):
    text = text.lower()
    for char in ".,;:!?¿¡()[]\"'":
        text = text.replace(char, "")
    return text.split()


def word2vec_sentence_embedding(text, word2vec_model):
    tokens = simple_tokenize_for_word2vec(text)
    vectors = []
    missing = []

    for token in tokens:
        if token in word2vec_model:
            vectors.append(word2vec_model[token])
        else:
            missing.append(token)

    if len(vectors) == 0:
        return None, missing

    return np.mean(vectors, axis=0), missing

## 3.2 Comparación preparada con Word2Vec

Ejecuta esta celda solo si has cargado `word2vec_model`.

In [44]:
# Ejecutar solo si has cargado word2vec_model

query_w2v, missing_query = word2vec_sentence_embedding(query, word2vec_model)
sentences_w2v = []
missing_by_sentence = {}

for sentence in sentences:
    emb, missing = word2vec_sentence_embedding(sentence, word2vec_model)
    sentences_w2v.append(emb)
    missing_by_sentence[sentence] = missing

w2v_results = build_similarity_table(query, sentences, query_w2v, sentences_w2v)
print("Palabras fuera de vocabulario en query:", missing_query)
print("Palabras fuera de vocabulario por frase:", missing_by_sentence)
w2v_results

Palabras fuera de vocabulario en query: []
Palabras fuera de vocabulario por frase: {'Dogs are pets.': [], 'This is a dog.': ['a'], 'They are free today.': []}


,Consulta,Frase comparada,Similitud coseno
0,Dogs are domestic animals.,Dogs are pets.,0.886546
1,Dogs are domestic animals.,This is a dog.,0.614273
2,Dogs are domestic animals.,They are free today.,0.408552


## 3.3 Comparación BERT vs SBERT

In [45]:
comparison_embeddings = pd.DataFrame({
    "Frase": sentences,
    "BERT mean pooling": [cosine_sim(query_bert_mean, emb) for emb in sentences_bert_mean],
    "BERT max pooling": [cosine_sim(query_bert_max, emb) for emb in sentences_bert_max],
    "SBERT MiniLM": [cosine_sim(query_minilm, emb) for emb in sentences_minilm],
    "SBERT MPNet": [cosine_sim(query_mpnet, emb) for emb in sentences_mpnet]
})

comparison_embeddings

,Frase,BERT mean pooling,BERT max pooling,SBERT MiniLM,SBERT MPNet
0,Dogs are pets.,0.817135,0.798324,0.848573,0.835124
1,This is a dog.,0.697745,0.811080,0.554901,0.482438
2,They are free today.,0.570568,0.758019,0.097927,0.068279


## Conclusión del Ejercicio 3

En general:

- **Word2Vec** puede funcionar si las frases comparten palabras, pero no entiende bien el contexto.
- **BERT** produce embeddings contextualizados, pero necesita pooling para representar frases.
- **SBERT** suele ser la mejor opción para similitud entre frases porque está entrenado para ello.

Con frases polisémicas habría más variación. Word2Vec mezcla sentidos porque el vector es estático. BERT y SBERT pueden distinguir mejor el significado según el contexto.

# Ejercicio 4. Medir similitud con diferentes embeddings

Probamos tres conjuntos de frases:

1. Frases con la misma idea pero distinto orden.
2. Frases con palabras polisémicas.
3. Frases parecidas superficialmente, pero con significado distinto.

Compararemos:

- BERT mean pooling.
- SBERT MiniLM.
- SBERT MPNet.

In [46]:
sentences_4_1 = [
    "Don´t shout at me, John.",
    "Don´t shout at John.",
    "John, stop shouting at me."
]

sentences_4_2 = [
    "The rolling Stones are rock idols.",
    "Don't throw me a rock.",
    "Iggy Pop is my favourite artist."
]

sentences_4_3 = [
    "I love the capital of Spain.",
    "I like Madrid.",
    "I love the capital of Portugal.",
    "I love the capital of Japan.",
    "I hate the capital of Spain.",
    "I hate Japan"
]

## 4.1 Función para comparar conjuntos de frases

In [47]:
def bert_mean_embeddings_for_sentences(sentence_list):
    return np.array([bert_sentence_embedding(sentence, pooling="mean")[0] for sentence in sentence_list])


def compare_sentence_set(sentence_list, title):
    print("=" * 120)
    print(title)
    print("=" * 120)

    bert_embs = bert_mean_embeddings_for_sentences(sentence_list)
    df_bert = pairwise_similarity_table(sentence_list, bert_embs)

    minilm_embs = sbert_minilm.encode(sentence_list)
    df_minilm = pairwise_similarity_table(sentence_list, minilm_embs)

    mpnet_embs = sbert_mpnet.encode(sentence_list)
    df_mpnet = pairwise_similarity_table(sentence_list, mpnet_embs)

    print("Matriz de similitud — BERT mean pooling")
    display(df_bert)

    print("Matriz de similitud — SBERT MiniLM")
    display(df_minilm)

    print("Matriz de similitud — SBERT MPNet")
    display(df_mpnet)

    return df_bert, df_minilm, df_mpnet

## 4.2 Caso 4.1 — Misma idea, diferente orden

Aquí hay que observar si los modelos distinguen quién realiza la acción y quién la recibe.

In [48]:
df_41_bert, df_41_minilm, df_41_mpnet = compare_sentence_set(
    sentences_4_1,
    "Caso 4.1 — Frases con orden diferente"
)

Caso 4.1 — Frases con orden diferente
Matriz de similitud — BERT mean pooling


,"Don´t shout at me, John.",Don´t shout at John.,"John, stop shouting at me."
"Don´t shout at me, John.",1.000000,0.674096,0.698578
Don´t shout at John.,0.674096,1.000000,0.486247
"John, stop shouting at me.",0.698578,0.486247,1.000000


Matriz de similitud — SBERT MiniLM


,"Don´t shout at me, John.",Don´t shout at John.,"John, stop shouting at me."
"Don´t shout at me, John.",1.000000,0.927670,0.858435
Don´t shout at John.,0.927670,1.000000,0.824004
"John, stop shouting at me.",0.858435,0.824004,1.000000


Matriz de similitud — SBERT MPNet


,"Don´t shout at me, John.",Don´t shout at John.,"John, stop shouting at me."
"Don´t shout at me, John.",1.000000,0.923923,0.913151
Don´t shout at John.,0.923923,1.000000,0.846483
"John, stop shouting at me.",0.913151,0.846483,1.000000


## 4.3 Caso 4.2 — Polisemia

La palabra `rock` puede significar música o piedra. Un buen modelo contextual debería distinguir esos usos.

In [49]:
df_42_bert, df_42_minilm, df_42_mpnet = compare_sentence_set(
    sentences_4_2,
    "Caso 4.2 — Polisemia"
)

Caso 4.2 — Polisemia
Matriz de similitud — BERT mean pooling


,The rolling Stones are rock idols.,Don't throw me a rock.,Iggy Pop is my favourite artist.
The rolling Stones are rock idols.,1.000000,0.566358,0.723245
Don't throw me a rock.,0.566358,1.000000,0.591167
Iggy Pop is my favourite artist.,0.723245,0.591167,1.000000


Matriz de similitud — SBERT MiniLM


,The rolling Stones are rock idols.,Don't throw me a rock.,Iggy Pop is my favourite artist.
The rolling Stones are rock idols.,1.000000,0.414100,0.437967
Don't throw me a rock.,0.414100,1.000000,0.156429
Iggy Pop is my favourite artist.,0.437967,0.156429,1.000000


Matriz de similitud — SBERT MPNet


,The rolling Stones are rock idols.,Don't throw me a rock.,Iggy Pop is my favourite artist.
The rolling Stones are rock idols.,1.000000,0.288622,0.387093
Don't throw me a rock.,0.288622,1.000000,0.119236
Iggy Pop is my favourite artist.,0.387093,0.119236,1.000000


## 4.4 Caso 4.3 — Frases parecidas pero significado distinto

Este caso es difícil porque frases como:

```text
I love the capital of Spain.
I hate the capital of Spain.
```

comparten casi todas las palabras, pero expresan sentimientos opuestos.

In [50]:
df_43_bert, df_43_minilm, df_43_mpnet = compare_sentence_set(
    sentences_4_3,
    "Caso 4.3 — Frases parecidas pero significado distinto"
)

Caso 4.3 — Frases parecidas pero significado distinto
Matriz de similitud — BERT mean pooling


,I love the capital of Spain.,I like Madrid.,I love the capital of Portugal.,I love the capital of Japan.,I hate the capital of Spain.,I hate Japan
I love the capital of Spain.,1.000000,0.794103,0.958938,0.920761,0.903746,0.631490
I like Madrid.,0.794103,1.000000,0.786142,0.763315,0.751312,0.705399
I love the capital of Portugal.,0.958938,0.786142,1.000000,0.915860,0.870228,0.634678
I love the capital of Japan.,0.920761,0.763315,0.915860,1.000000,0.848311,0.677504
I hate the capital of Spain.,0.903746,0.751312,0.870228,0.848311,1.000000,0.738428
I hate Japan,0.631490,0.705399,0.634678,0.677504,0.738428,1.000000


Matriz de similitud — SBERT MiniLM


,I love the capital of Spain.,I like Madrid.,I love the capital of Portugal.,I love the capital of Japan.,I hate the capital of Spain.,I hate Japan
I love the capital of Spain.,1.000000,0.611117,0.751943,0.536329,0.814957,0.266910
I like Madrid.,0.611117,1.000000,0.459537,0.220976,0.533869,0.275336
I love the capital of Portugal.,0.751943,0.459537,1.000000,0.541381,0.582756,0.263628
I love the capital of Japan.,0.536329,0.220976,0.541381,1.000001,0.407424,0.650912
I hate the capital of Spain.,0.814957,0.533869,0.582756,0.407424,1.000000,0.501613
I hate Japan,0.266910,0.275336,0.263628,0.650912,0.501613,1.000000


Matriz de similitud — SBERT MPNet


,I love the capital of Spain.,I like Madrid.,I love the capital of Portugal.,I love the capital of Japan.,I hate the capital of Spain.,I hate Japan
I love the capital of Spain.,1.000000,0.641955,0.808544,0.707466,0.724722,0.300713
I like Madrid.,0.641955,1.000000,0.570794,0.376126,0.503369,0.235913
I love the capital of Portugal.,0.808544,0.570794,1.000000,0.710199,0.569259,0.313899
I love the capital of Japan.,0.707466,0.376126,0.710199,1.000000,0.468929,0.623705
I hate the capital of Spain.,0.724722,0.503369,0.569259,0.468929,1.000000,0.491789
I hate Japan,0.300713,0.235913,0.313899,0.623705,0.491789,1.000000


## Conclusión del Ejercicio 4

Lo normal es observar que SBERT funciona mejor para similitud de frases que BERT con pooling manual.

Sin embargo, incluso SBERT puede fallar en frases muy parecidas superficialmente pero con significado contrario.

Los casos de polisemia muestran mejor la ventaja de los embeddings contextualizados frente a embeddings estáticos como Word2Vec.

# Conclusión general

Resumen final:

| Método | Tipo | Ventaja | Limitación |
|---|---|---|---|
| Word2Vec | Estático | Simple y rápido | No entiende contexto ni polisemia |
| BERT | Contextual por token | Captura contexto | Necesita pooling para frases |
| SBERT | Contextual de frase | Muy bueno para similitud semántica | Depende del modelo elegido |

Para similitud entre frases, normalmente:

```text
SBERT > BERT con pooling manual > Word2Vec promedio
```

Especialmente cuando hay polisemia, cambios de orden o diferencias semánticas sutiles.